# RQ2 — VOV4 Media Probe

Notebook này đọc 30 bài đã chọn ở Notebook 07 và ghi lại media URL **xuất hiện sẵn** trong HTML công khai.

- Không tải file audio/video.
- Không đoán URL, không gọi API nội bộ, không bypass token, DRM hay anti-bot.
- Không pseudo-label, không ASR/MT, không training.
- Không mở frozen test, không sửa artifact RQ1 hay artifact Notebook 07.

Bước sau, khi probe này PASS, Notebook 09 mới được tải thử một lượng nhỏ.


## 1. Imports / config

Chạy bằng venv `bahnar-s2tt` trên macOS. `VERIFY_MEDIA=True` chỉ gửi HEAD, hoặc GET `Range: bytes=0-0` khi server không hỗ trợ HEAD.


In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("bs4") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "beautifulsoup4"])

from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse, urlunparse
from urllib.robotparser import RobotFileParser
import hashlib
import ipaddress
import socket
import json

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display


def find_project_root() -> Path:
    node = Path.cwd().resolve()
    for cand in [node, *node.parents]:
        if (cand / "requirements.txt").is_file() and (cand / "notebooks").is_dir() and (cand / "src").is_dir():
            return cand
    raise RuntimeError(f"Cannot locate bahnar-s2tt-thesis root from {node}")


PROJECT_ROOT = find_project_root()
DISCOVERY_DIR = PROJECT_ROOT / "artifacts" / "rq2" / "vov4_discovery"
CANDIDATES_CSV = DISCOVERY_DIR / "vov4_candidates.csv"
DISCOVERY_SUMMARY = DISCOVERY_DIR / "summary.json"
RAW_HTML_DIR = DISCOVERY_DIR / "raw_html"
OUT_DIR = PROJECT_ROOT / "artifacts" / "rq2" / "vov4_media_probe"

EXPECTED_ROWS = 30
SLEEP_SECONDS = 1.5
VERIFY_MEDIA = True
VERIFY_ATTEMPTS = 3
VERIFY_BACKOFF_SECONDS = (2, 4, 8)
MEDIA_EXTENSIONS = (".m3u8", ".m4a", ".mp3", ".mp4", ".aac", ".wav")
AUDIO_EXTENSIONS = {"mp3", "aac", "m4a", "wav"}
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp", ".gif")
MAX_REDIRECTS = 5
STRONG_MEDIA_KEYS = {"file", "audio", "video", "media", "playlist", "stream", "sources"}
GENERIC_MEDIA_KEYS = {"src", "source", "contenturl", "embedurl"}
DATA_URL_ATTRS = {
    "data-url", "data-src", "data-file", "data-audio",
    "data-video", "data-stream", "data-playlist",
}

USER_AGENT = (
    "BahnarS2TT-Thesis-Research/1.0 "
    "(+non-commercial academic media probe; no bulk download)"
)

session = requests.Session()
session.headers.update({
    "User-Agent": USER_AGENT,
    "Accept-Language": "vi,en;q=0.8",
})

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CANDIDATES_CSV:", CANDIDATES_CSV)
print("OUT_DIR:", OUT_DIR)
print("VERIFY_MEDIA:", VERIFY_MEDIA)


PROJECT_ROOT: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis
CANDIDATES_CSV: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/artifacts/rq2/vov4_discovery/vov4_candidates.csv
OUT_DIR: /Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/artifacts/rq2/vov4_media_probe
VERIFY_MEDIA: True


/Users/minhtuan25/Desktop/MinhTuanCode/MasterAI/citizen-assistance-system/bahnar-s2tt-thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Load + validate Notebook 07 artifacts

Chỉ đọc `vov4_candidates.csv` và `summary.json`. Không ghi vào thư mục discovery.


In [2]:
if not CANDIDATES_CSV.is_file():
    raise RuntimeError(f"Missing Notebook 07 candidates: {CANDIDATES_CSV}")

candidates = pd.read_csv(CANDIDATES_CSV, dtype=str).fillna("")
if len(candidates) != EXPECTED_ROWS:
    raise RuntimeError(f"Expected {EXPECTED_ROWS} discovery rows, found {len(candidates)}")
if not candidates["source"].eq("VOV4").all():
    raise RuntimeError("source phải là VOV4")
if candidates["source_id"].duplicated().any():
    raise RuntimeError("source_id bị trùng")
if candidates["page_url"].duplicated().any():
    raise RuntimeError("page_url bị trùng")
for column in ("source_id", "page_url", "title", "published_date", "candidate_priority", "page_sha256"):
    if column not in candidates.columns:
        raise RuntimeError(f"Candidates CSV missing column {column}")

if DISCOVERY_SUMMARY.is_file():
    discovery_summary = json.loads(DISCOVERY_SUMMARY.read_text(encoding="utf-8"))
    if discovery_summary.get("status") != "SUCCESS_VOV4_DISCOVERY":
        raise RuntimeError(f"Notebook 07 summary is not PASS: {discovery_summary.get('status')}")
    if int(discovery_summary.get("n_fetch_failed", 1)) != 0:
        raise RuntimeError("Notebook 07 still has fetch failures")
    if int(discovery_summary.get("n_candidates_final", -1)) != EXPECTED_ROWS:
        raise RuntimeError("Notebook 07 n_candidates_final is not 30")
else:
    discovery_summary = None
    print("WARN: summary.json không có; chỉ kiểm tra CSV")

print("candidates:", len(candidates))
candidates[["source_id", "published_date", "candidate_priority", "title"]].head(3)


candidates: 30


,source_id,published_date,candidate_priority,title
0,VOV4_23A5518B701085121391274F43D7F0A6854AF4A5,2026-09-24,normal,Tơdrong kơtơ̆ng ang năr 24.9.2026
1,VOV4_2D39DE6994374745A90B2F8CCC41A43ABF111EF0,2026-09-24,normal,Chánh án Hơnih tơm xek tơlang teh đak Nguyễn V...
2,VOV4_4F3554FC564400008578B69EB55E4D529F2A6E0E,2026-09-24,normal,Hop akŏm pơdrơ̆ng kiơ̆ trong chih tơlĕch Hla b...


## 3. robots / safe request helpers

Request lại HTML bài chỉ khi file raw bị thiếu, và chỉ trên `vov4.vov.vn`. Media URL có thể ở host khác nếu nó xuất hiện trong HTML. Mỗi hop đều kiểm tra `robots.txt` trước khi gửi request. Không `allow_redirects=True`.


In [3]:
def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).strftime("%Y-%m-%dT%H:%M:%SZ")


def canonicalize(base_url: str, raw_url: str) -> str:
    absolute = urljoin(base_url, (raw_url or "").strip())
    parts = urlparse(absolute)
    if parts.scheme not in {"http", "https"}:
        return ""
    return urlunparse((
        parts.scheme.lower(),
        parts.netloc.lower(),
        parts.path or "/",
        "",
        parts.query,
        "",
    ))


import time


class RobotsUnverified(RuntimeError):
    pass


class NonPublicHost(RuntimeError):
    pass


ROBOTS = {}
PUBLIC_HOSTS = {}


def assert_public_http_host(url: str) -> None:
    """Reject localhost, .local, and any non-global address. Fail closed."""
    host = (urlparse(url).hostname or "").lower().rstrip(".")
    if not host:
        raise NonPublicHost(f"host không hợp lệ: {url}")
    if host in PUBLIC_HOSTS:
        if PUBLIC_HOSTS[host] is not None:
            raise NonPublicHost(PUBLIC_HOSTS[host])
        return
    if host in {"localhost", "localhost.localdomain"} or host.endswith(".localhost") or host.endswith(".local"):
        message = f"local host bị chặn: {host}"
        PUBLIC_HOSTS[host] = message
        raise NonPublicHost(message)
    addresses = []
    try:
        addresses.append(ipaddress.ip_address(host))
    except ValueError:
        try:
            infos = socket.getaddrinfo(host, None)
        except socket.gaierror as exc:
            message = f"không resolve được {host}: {exc}"
            PUBLIC_HOSTS[host] = message
            raise NonPublicHost(message) from exc
        if not infos:
            message = f"không resolve được {host}"
            PUBLIC_HOSTS[host] = message
            raise NonPublicHost(message)
        for info in infos:
            addresses.append(ipaddress.ip_address(info[4][0].split("%", 1)[0]))
    for ip in addresses:
        if (
            ip.is_private or ip.is_loopback or ip.is_link_local or ip.is_multicast
            or ip.is_reserved or ip.is_unspecified or not ip.is_global
        ):
            message = f"non-public address bị chặn: {host} -> {ip}"
            PUBLIC_HOSTS[host] = message
            raise NonPublicHost(message)
    PUBLIC_HOSTS[host] = None


def robots_for(url: str) -> RobotFileParser:
    """Load robots.txt for this exact host. Fail closed if it cannot be verified."""
    parts = urlparse(url)
    if parts.scheme not in {"http", "https"} or not parts.netloc:
        raise RobotsUnverified(f"URL không phải http(s): {url}")
    assert_public_http_host(url)
    origin = f"{parts.scheme}://{parts.netloc}"
    if origin in ROBOTS:
        return ROBOTS[origin]
    robots_url = origin + "/robots.txt"
    time.sleep(SLEEP_SECONDS)
    response = session.get(robots_url, timeout=20, allow_redirects=False)
    try:
        if response.status_code in {301, 302, 303, 307, 308}:
            raise RobotsUnverified(f"robots.txt redirect, không xác minh được: {robots_url}")
        if response.status_code != 200 or "user-agent" not in response.text.lower():
            raise RobotsUnverified(
                f"Không xác minh được robots.txt: {robots_url} HTTP {response.status_code}"
            )
        parser = RobotFileParser()
        parser.set_url(robots_url)
        parser.parse(response.text.splitlines())
    finally:
        response.close()
    ROBOTS[origin] = parser
    return parser


def robots_decision(url: str) -> str:
    parser = robots_for(url)
    return "allow" if parser.can_fetch(USER_AGENT, url) else "disallow"


def follow_hops(url: str, *, method: str, host_lock: str = ""):
    """Manual redirects only. Robots is checked before every request."""
    current = canonicalize(url, url)
    seen = []
    for _hop in range(MAX_REDIRECTS):
        if not current:
            raise RuntimeError(f"URL không hợp lệ: {url}")
        if host_lock and urlparse(current).netloc.lower() != host_lock:
            raise RuntimeError(f"Redirect rời {host_lock}: {current}")
        if current in seen:
            raise RuntimeError(f"Redirect loop: {current}")
        seen.append(current)
        try:
            assert_public_http_host(current)
            decision = robots_decision(current)
        except NonPublicHost as exc:
            return {
                "outcome": "protected",
                "notes": str(exc),
                "final_url": current,
                "http_status": "",
                "content_type": "",
                "content_length": "",
                "accepts_ranges": "",
                "body": b"",
            }
        except RobotsUnverified as exc:
            return {
                "outcome": "protected",
                "notes": str(exc),
                "final_url": current,
                "http_status": "",
                "content_type": "",
                "content_length": "",
                "accepts_ranges": "",
                "body": b"",
            }
        if decision == "disallow":
            return {
                "outcome": "protected",
                "notes": f"robots.txt disallow: {current}",
                "final_url": current,
                "http_status": "",
                "content_type": "",
                "content_length": "",
                "accepts_ranges": "",
                "body": b"",
            }
        time.sleep(SLEEP_SECONDS)
        if method == "GET":
            response = session.get(current, timeout=30, allow_redirects=False)
        else:
            response = session.head(current, timeout=20, allow_redirects=False)
            if response.status_code in {405, 501}:
                response.close()
                response = session.get(
                    current,
                    timeout=20,
                    headers={"Range": "bytes=0-0"},
                    stream=True,
                    allow_redirects=False,
                )
        try:
            if response.status_code in {301, 302, 303, 307, 308}:
                location = response.headers.get("Location") or ""
                if not location.strip():
                    raise RuntimeError(f"Redirect thiếu Location: {current}")
                current = canonicalize(current, location.strip())
                continue
            payload = response.content if method == "GET" else b""
            result = {
                "outcome": "ok",
                "final_url": current,
                "http_status": str(response.status_code),
                "content_type": (response.headers.get("Content-Type") or "").split(";")[0].strip().lower(),
                "content_length": response.headers.get("Content-Length") or "",
                "accepts_ranges": response.headers.get("Accept-Ranges") or "",
                "body": payload,
                "notes": "",
            }
            return result
        finally:
            response.close()
    raise RuntimeError(f"Quá {MAX_REDIRECTS} redirect hops: {url}")


def fetch_page_html(url: str) -> bytes:
    if urlparse(url).netloc.lower() != "vov4.vov.vn":
        raise RuntimeError(f"Refetch host is not vov4.vov.vn: {url}")
    result = follow_hops(url, method="GET", host_lock="vov4.vov.vn")
    if result["outcome"] != "ok":
        raise PermissionError(result["notes"])
    ctype = result["content_type"]
    if "text/html" not in ctype and "application/xhtml" not in ctype:
        raise RuntimeError(f"Không phải HTML: {ctype}")
    return result["body"]


def load_article_html(row) -> tuple:
    path = RAW_HTML_DIR / f"{row['source_id']}.html"
    if path.is_file():
        return path.read_bytes(), "raw_html"
    return fetch_page_html(row["page_url"]), "refetch"


## 4. Media extraction helpers

Chỉ lấy URL nằm trong article chính: `article[role=article][about=<pathname của page_url>]`. Related, sidebar, header, footer không được gán cho bài hiện tại. `data-*` chỉ đọc attribute URL (`data-url`, `data-src`, `data-file`, `data-audio`, `data-video`, `data-stream`, `data-playlist`).


In [4]:
import re

INLINE_URL_RE = re.compile(
    "(?P<q>[\"'])(?P<url>(?:https?:)?//[^\\s\"'<>]+|/[^\\s\"'<>]+)(?P=q)"
)
INLINE_ASSIGN_RE = re.compile(
    "(?<![A-Za-z0-9_])"
    "(?P<key>sources|source|playlist|stream|audio|video|media|file|src)"
    r"\s*[:=]\s*"
    "(?P<q>[\"'])"
    r"(?P<url>[^\"'\s<>]+)"
    "(?P=q)",
    re.IGNORECASE,
)

def media_extension(url: str) -> str:
    path = urlparse(url).path.lower()
    for ext in MEDIA_EXTENSIONS:
        if path.endswith(ext):
            return ext[1:]
    return ""


def is_image_url(url: str) -> bool:
    path = urlparse(url).path.lower()
    return any(path.endswith(ext) for ext in IMAGE_EXTENSIONS)


def looks_like_url_ref(raw: str) -> bool:
    value = (raw or "").strip()
    if not value or value.lower().startswith(("data:", "javascript:", "blob:")):
        return False
    lowered = value.lower().split("?", 1)[0].split("#", 1)[0]
    if any(lowered.endswith(ext) for ext in IMAGE_EXTENSIONS):
        return False
    if value.startswith(("http://", "https://", "//", "/", "./", "../")):
        return True
    if "/" in value:
        return True
    return any(lowered.endswith(ext) for ext in MEDIA_EXTENSIONS)


def media_type_for(extension: str, hint: str = "") -> str:
    if extension == "m3u8":
        if hint == "audio":
            return "audio"
        if hint == "video":
            return "video"
        return "unknown_stream"
    if hint in {"audio", "video"}:
        return hint
    if extension in AUDIO_EXTENSIONS:
        return "audio"
    if extension == "mp4":
        return "video"
    return ""


def consider_url(found: dict, page_url: str, raw_url: str, method: str, hint: str = "", strong: bool = False) -> None:
    raw_url = (raw_url or "").strip()
    if not raw_url:
        return
    media_url = canonicalize(page_url, raw_url)
    if not media_url or is_image_url(media_url):
        return
    if method in {"audio_tag", "video_tag", "og_audio", "og_video"}:
        strong = True
    elif method == "source_tag":
        strong = strong or hint in {"audio", "video"}
    elif method == "href_src_scan":
        strong = False
    extension = media_extension(media_url)
    if not extension and not strong:
        return
    kind = media_type_for(extension, hint)
    if not kind and strong:
        kind = "unknown_stream"
    if not kind:
        return
    if media_url in found:
        return
    parts = urlparse(media_url)
    found[media_url] = {
        "media_url": media_url,
        "media_type": kind,
        "media_extension": extension,
        "media_host": parts.netloc.lower(),
        "discovery_method": method,
        "has_query_token": bool(parts.query),
        "is_public_direct_url": False,
    }


def walk_json(node, bucket: list, path: tuple = ()) -> None:
    if isinstance(node, dict):
        raw_type = node.get("@type", node.get("type", ""))
        if isinstance(raw_type, list):
            type_name = " ".join(str(item) for item in raw_type).lower()
        elif isinstance(raw_type, str):
            type_name = raw_type.lower()
        else:
            type_name = ""
        mime = ""
        for child_key, value in node.items():
            if str(child_key).lower() in {"encodingformat", "mime", "mimetype", "contenttype"} and isinstance(value, str):
                mime = value.lower()
        for child_key, value in node.items():
            key = str(child_key).lower()
            walk_json(value, bucket, path + ((key, type_name, mime),))
    elif isinstance(node, list):
        for value in node:
            walk_json(value, bucket, path)
    elif isinstance(node, str):
        bucket.append((path, node))


def json_media_decision(path: tuple) -> tuple:
    if not path:
        return False, ""
    key, type_name, mime = path[-1]
    if not mime and (type_name.startswith("audio/") or type_name.startswith("video/")):
        mime = type_name
    parent_keys = " ".join(item[0] for item in path[:-1])
    hint = ""
    if mime.startswith("audio/") or "audioobject" in type_name or key == "audio":
        hint = "audio"
    elif mime.startswith("video/") or "videoobject" in type_name or key == "video":
        hint = "video"
    if not hint and "audio" in parent_keys and "video" not in parent_keys:
        hint = "audio"
    elif not hint and "video" in parent_keys and "audio" not in parent_keys:
        hint = "video"
    parent_context = any(token in parent_keys for token in ("audio", "video", "player", "media", "stream"))
    object_context = "audioobject" in type_name or "videoobject" in type_name
    mime_context = mime.startswith("audio/") or mime.startswith("video/") or mime in {
        "application/vnd.apple.mpegurl", "application/x-mpegurl",
    }
    strong = key in STRONG_MEDIA_KEYS or object_context or parent_context or mime_context
    return strong, hint


def add_json_value(found: dict, page_url: str, path: tuple, value: str, method: str) -> None:
    if not path or not looks_like_url_ref(value):
        return
    key = path[-1][0]
    if key in {"@type", "type", "encodingformat", "mime", "mimetype", "contenttype"}:
        return
    strong, hint = json_media_decision(path)
    consider_url(found, page_url, value, method, hint, strong=strong)


def inline_statement(script_text: str, start: int) -> str:
    chunk = script_text[max(0, start - 80):start]
    for sep in ("\n", ";", "{"):
        idx = chunk.rfind(sep)
        if idx != -1:
            chunk = chunk[idx + 1:]
    return chunk.lower()


def inline_has_media_context(script_text: str, start: int, end: int) -> bool:
    window = inline_statement(script_text, start)
    return any(token in window for token in ("audio", "video", "player", "media", "stream"))


def extract_inline_script(script_text: str, page_url: str, found: dict) -> None:
    if not script_text:
        return
    for match in INLINE_ASSIGN_RE.finditer(script_text):
        raw = match.group("url")
        if not looks_like_url_ref(raw):
            continue
        key = match.group("key").lower()
        hint = "audio" if key == "audio" else "video" if key == "video" else ""
        contextual = inline_has_media_context(script_text, match.start(), match.end())
        if not hint and key in {"src", "source"}:
            window = inline_statement(script_text, match.start())
            if "audio" in window and "video" not in window:
                hint = "audio"
            elif "video" in window and "audio" not in window:
                hint = "video"
        strong = key in STRONG_MEDIA_KEYS or (key in {"src", "source"} and contextual)
        consider_url(found, page_url, raw, "inline_script", hint, strong=strong)
    for match in INLINE_URL_RE.finditer(script_text):
        raw = match.group("url")
        if not looks_like_url_ref(raw):
            continue
        absolute = canonicalize(page_url, raw)
        if absolute and media_extension(absolute):
            consider_url(found, page_url, raw, "inline_script", "", strong=False)


def find_primary_article(soup: BeautifulSoup, page_url: str):
    pathname = urlparse(page_url).path
    if not pathname:
        return None
    matches = [
        node for node in soup.find_all("article")
        if node.get("role") == "article" and (node.get("about") or "") == pathname
    ]
    if len(matches) != 1:
        return None
    return matches[0]


def extract_media(root, page_url: str) -> list:
    found = {}
    for tag in root.find_all("audio"):
        if tag.get("src"):
            consider_url(found, page_url, tag.get("src"), "audio_tag", "audio")
        for source in tag.find_all("source"):
            raw = source.get("src") or source.get("data-src")
            if raw:
                consider_url(found, page_url, raw, "source_tag", "audio")
    for tag in root.find_all("video"):
        if tag.get("src"):
            consider_url(found, page_url, tag.get("src"), "video_tag", "video")
        for source in tag.find_all("source"):
            raw = source.get("src") or source.get("data-src")
            if raw:
                consider_url(found, page_url, raw, "source_tag", "video")
    for source in root.find_all("source"):
        if source.find_parent(["audio", "video"]) is not None:
            continue
        raw = source.get("src") or source.get("data-src")
        mime = (source.get("type") or "").lower()
        hint = "audio" if mime.startswith("audio/") else "video" if mime.startswith("video/") else ""
        hls_mime = mime in {"application/vnd.apple.mpegurl", "application/x-mpegurl"}
        if raw:
            consider_url(found, page_url, raw, "source_tag", hint, strong=hls_mime)

    for prop, method, hint in (
        ("og:audio", "og_audio", "audio"),
        ("og:video", "og_video", "video"),
    ):
        for node in root.select(f'meta[property="{prop}"]'):
            if node.get("content"):
                consider_url(found, page_url, node.get("content"), method, hint)

    for script in root.find_all("script", attrs={"type": "application/ld+json"}):
        pairs = []
        try:
            walk_json(json.loads(script.string or ""), pairs)
        except (TypeError, json.JSONDecodeError):
            continue
        for path, value in pairs:
            add_json_value(found, page_url, path, value, "json_ld")

    for script in root.find_all("script", attrs={"type": "application/json"}):
        pairs = []
        try:
            walk_json(json.loads(script.string or ""), pairs)
        except (TypeError, json.JSONDecodeError):
            continue
        for path, value in pairs:
            add_json_value(found, page_url, path, value, "script_config")

    for script in root.find_all("script"):
        script_type = (script.get("type") or "").lower()
        if script_type in {"application/json", "application/ld+json"}:
            continue
        extract_inline_script(script.string or "", page_url, found)

    for tag in root.find_all(True):
        classes = " ".join(tag.get("class") or []).lower()
        hint = "audio" if "audio" in classes else "video" if "video" in classes else ""
        for key, value in tag.attrs.items():
            if str(key).lower() not in DATA_URL_ATTRS or not isinstance(value, str):
                continue
            attr_key = str(key)[5:].lower()
            strong = hint in {"audio", "video"} or attr_key in {"audio", "video", "media", "playlist", "stream"}
            consider_url(found, page_url, value, "data_attribute", hint, strong=strong)

    for tag in root.find_all(True):
        for attr in ("src", "href"):
            value = tag.get(attr)
            if isinstance(value, str):
                consider_url(found, page_url, value, "href_src_scan")
    return list(found.values())


## 5. Parse raw HTML

Ưu tiên HTML Notebook 07 đã lưu. Đối chiếu SHA256 với `page_sha256` trước khi parse. Chỉ extract trong article chính. Không thấy đúng một `article[about=pathname]` thì `ERROR_ARTICLE_SCOPE_NOT_FOUND` và không quét cả trang.


In [5]:
article_records = []
for row in candidates.to_dict(orient="records"):
    record = {
        "source_id": row["source_id"],
        "page_url": row["page_url"],
        "title": row["title"],
        "published_date": row["published_date"],
        "candidate_priority": row["candidate_priority"],
        "media": [],
        "error_type": "",
        "error_message": "",
    }
    try:
        html_bytes, html_source = load_article_html(row)
        expected_sha = (row.get("page_sha256") or "").strip().lower()
        digest = hashlib.sha256(html_bytes).hexdigest()
        if not expected_sha or digest != expected_sha:
            raise RuntimeError(f"page_sha256 mismatch: got {digest}")
        soup = BeautifulSoup(html_bytes, "html.parser")
        article = find_primary_article(soup, row["page_url"])
        if article is None:
            pathname = urlparse(row["page_url"]).path
            record["error_type"] = "ERROR_ARTICLE_SCOPE_NOT_FOUND"
            record["error_message"] = f"Không thấy đúng một article about={pathname}"
        else:
            record["media"] = extract_media(article, row["page_url"])
        record["html_source"] = html_source
    except Exception as exc:
        record["error_type"] = type(exc).__name__
        record["error_message"] = str(exc)
        record["html_source"] = "error"
        print("WARN:", row["source_id"], type(exc).__name__, str(exc))
    article_records.append(record)

print("parsed:", len(article_records))
print("with_media:", sum(1 for item in article_records if item["media"]))
print("errors:", sum(1 for item in article_records if item["error_type"]))


parsed: 30
with_media: 15
errors: 0


## 6. Optional safe verification

HEAD trước, `allow_redirects=False`, tối đa 5 hop. Mỗi hop được canonicalize, kiểm tra http/https và `robots.txt` của đúng host đó trước khi gửi request. Chỉ GET `Range: bytes=0-0` khi HEAD trả 405 hoặc 501, `stream=True`, rồi đóng ngay. `is_public_direct_url` chỉ được True sau HTTP 200/206 với Content-Type audio, video hoặc HLS.


In [6]:
HLS_TYPES = {"application/vnd.apple.mpegurl", "application/x-mpegurl"}
BINARY_TYPES = {"application/octet-stream", "binary/octet-stream"}


def apply_content_type(media: dict, ctype: str) -> None:
    if ctype.startswith("audio/"):
        media["media_type"] = "audio"
    elif ctype.startswith("video/"):
        media["media_type"] = "video"
    elif ctype in HLS_TYPES and media["media_type"] not in {"audio", "video"}:
        media["media_type"] = "unknown_stream"


def content_type_is_media(ctype: str, media_type: str) -> bool:
    if ctype.startswith("audio/") or ctype.startswith("video/") or ctype in HLS_TYPES:
        return True
    return ctype in BINARY_TYPES and media_type in {"audio", "video", "unknown_stream"}


def content_type_is_non_media(ctype: str) -> bool:
    if ctype.startswith("image/") or ctype.startswith("font/"):
        return True
    return ctype in {
        "text/css",
        "text/javascript",
        "application/javascript",
        "application/x-javascript",
        "application/font-woff",
        "application/x-font-ttf",
        "application/x-font-woff",
    }


def _is_transient_verify_error(exc: BaseException) -> bool:
    return isinstance(exc, (requests.exceptions.Timeout, requests.exceptions.ConnectionError))


def _verify_status_is_transient(result: dict) -> bool:
    try:
        code = int(result.get("http_status") or 0)
    except (TypeError, ValueError):
        return False
    return code == 429 or 500 <= code <= 599


def verify_one(media: dict) -> None:
    url = media["media_url"]
    media.update({
        "http_status": "",
        "content_type": "",
        "content_length": "",
        "accepts_ranges": "",
        "probe_status": "MEDIA_FOUND_UNVERIFIED",
        "is_public_direct_url": False,
        "notes": "",
    })
    if not VERIFY_MEDIA:
        media["notes"] = "verification skipped"
        return
    result = None
    for attempt in range(1, VERIFY_ATTEMPTS + 1):
        try:
            result = follow_hops(url, method="HEAD")
        except Exception as exc:
            if not _is_transient_verify_error(exc) or attempt >= VERIFY_ATTEMPTS:
                media["probe_status"] = "ERROR"
                media["notes"] = f"{type(exc).__name__}: {exc}"
                return
            delay = VERIFY_BACKOFF_SECONDS[attempt - 1]
            print(f"RETRY {attempt}/{VERIFY_ATTEMPTS - 1} in {delay}s: {type(exc).__name__} {url}")
            time.sleep(delay)
            continue
        if result.get("outcome") == "protected" or not _verify_status_is_transient(result) or attempt >= VERIFY_ATTEMPTS:
            break
        delay = VERIFY_BACKOFF_SECONDS[attempt - 1]
        print(f"RETRY {attempt}/{VERIFY_ATTEMPTS - 1} in {delay}s: HTTP {result.get('http_status')} {url}")
        time.sleep(delay)
        result = None
    if result.get("outcome") == "protected":
        media["probe_status"] = "PROTECTED_OR_UNSUPPORTED"
        media["notes"] = result.get("notes") or "robots.txt unverified or disallow"
        return

    media["http_status"] = result.get("http_status") or ""
    media["content_type"] = result.get("content_type") or ""
    media["content_length"] = result.get("content_length") or ""
    media["accepts_ranges"] = result.get("accepts_ranges") or ""
    final_url = result.get("final_url") or url
    if final_url != url:
        media["notes"] = f"redirected to {final_url}"
    apply_content_type(media, media["content_type"])

    code = int(media["http_status"] or 0)
    ctype = media["content_type"]
    if code in {401, 403}:
        media["probe_status"] = "PROTECTED_OR_UNSUPPORTED"
        return
    if ctype.startswith("text/html"):
        media["probe_status"] = "PROTECTED_OR_UNSUPPORTED"
        media["notes"] = (media["notes"] + "; " if media["notes"] else "") + "response is HTML, not a direct media file"
        return
    if content_type_is_non_media(ctype):
        media["probe_status"] = "NON_MEDIA"
        media["media_found"] = False
        media["is_public_direct_url"] = False
        media["notes"] = (media["notes"] + "; " if media["notes"] else "") + f"content-type is not media: {ctype}"
        return
    if code in {200, 206} and content_type_is_media(ctype, media["media_type"]):
        media["probe_status"] = "MEDIA_FOUND_PUBLIC"
        media["is_public_direct_url"] = True
        return
    media["probe_status"] = "MEDIA_FOUND_UNVERIFIED"
    extra = f"unverified HTTP {code}" + (f" content-type {ctype}" if ctype else "")
    media["notes"] = (media["notes"] + "; " if media["notes"] else "") + extra


verified = 0
for record in article_records:
    for media in record["media"]:
        verify_one(media)
        verified += 1
print("verified_urls:", verified)


verified_urls: 15


## 7. Build media manifest

`vov4_media_probe.csv` giữ mỗi quan hệ bài–media, nên cùng một media URL có thể xuất hiện ở nhiều `source_id`. `media_candidates.jsonl` dedupe theo canonical `media_url` và ghi `source_ids` / `page_urls` đã tham chiếu nó.


In [7]:
PROBE_COLUMNS = [
    "source_id", "page_url", "title", "published_date", "candidate_priority",
    "media_found", "media_type", "media_url", "media_host", "media_extension",
    "discovery_method", "is_public_direct_url", "has_query_token", "probe_status",
    "http_status", "content_type", "content_length", "accepts_ranges", "notes",
    "shared_source_count", "shared_media_url",
]

probe_rows = []
for record in article_records:
    base = {
        "source_id": record["source_id"],
        "page_url": record["page_url"],
        "title": record["title"],
        "published_date": record["published_date"],
        "candidate_priority": record["candidate_priority"],
    }
    empty = {
        "media_found": False,
        "media_type": "",
        "media_url": "",
        "media_host": "",
        "media_extension": "",
        "discovery_method": "",
        "is_public_direct_url": False,
        "has_query_token": False,
        "http_status": "",
        "content_type": "",
        "content_length": "",
        "accepts_ranges": "",
        "shared_source_count": 0,
        "shared_media_url": False,
    }
    if record["error_type"]:
        status_name = (
            "ERROR_ARTICLE_SCOPE_NOT_FOUND"
            if record["error_type"] == "ERROR_ARTICLE_SCOPE_NOT_FOUND"
            else "ERROR"
        )
        probe_rows.append({
            **base,
            **empty,
            "probe_status": status_name,
            "notes": f"{record['error_type']}: {record['error_message']}",
        })
        continue
    if not record["media"]:
        probe_rows.append({**base, **empty, "probe_status": "NO_MEDIA_FOUND", "notes": ""})
        continue
    for media in record["media"]:
        probe_rows.append({**base, "media_found": True, **media})

def build_media_candidates(rows: list) -> list:
    grouped = {}
    for row in rows:
        media_url = row["media_url"]
        if (
            not media_url
            or not row.get("media_found")
            or row.get("probe_status") == "NON_MEDIA"
        ):
            continue
        item = grouped.get(media_url)
        if item is None:
            item = {
                "media_url": media_url,
                "media_type": row["media_type"],
                "media_extension": row["media_extension"],
                "media_host": row["media_host"],
                "discovery_method": row["discovery_method"],
                "is_public_direct_url": bool(row["is_public_direct_url"]),
                "has_query_token": bool(row["has_query_token"]),
                "probe_status": row["probe_status"],
                "http_status": row["http_status"],
                "content_type": row["content_type"],
                "content_length": row["content_length"],
                "accepts_ranges": row["accepts_ranges"],
                "notes": row["notes"],
                "source_ids": [],
                "page_urls": [],
            }
            grouped[media_url] = item
        if row["source_id"] not in item["source_ids"]:
            item["source_ids"].append(row["source_id"])
            item["page_urls"].append(row["page_url"])
    media_candidates = list(grouped.values())
    for item in media_candidates:
        if len(item["source_ids"]) != len(item["page_urls"]):
            raise RuntimeError(f"source_ids/page_urls lệch nhau: {item['media_url']}")
        item["shared_source_count"] = len(item["source_ids"])
        item["shared_media_url"] = item["shared_source_count"] > 1
    return media_candidates


def _synthetic_shared_provenance() -> None:
    base = {
        "media_type": "audio",
        "media_extension": "mp3",
        "media_host": "example.com",
        "discovery_method": "data_attribute",
        "is_public_direct_url": True,
        "has_query_token": False,
        "probe_status": "MEDIA_FOUND_PUBLIC",
        "http_status": "200",
        "content_type": "audio/mpeg",
        "content_length": "1",
        "accepts_ranges": "bytes",
        "notes": "",
        "media_found": True,
        "media_url": "https://example.com/shared.mp3",
    }
    probe_rows_test = [
        {**base, "source_id": "A", "page_url": "https://example.com/a"},
        {**base, "source_id": "B", "page_url": "https://example.com/b"},
        {**base, "source_id": "A", "page_url": "https://example.com/a-again"},
    ]
    candidates = build_media_candidates(probe_rows_test)
    if len(candidates) != 1:
        raise AssertionError(f"expected 1 candidate, got {len(candidates)}")
    candidate = candidates[0]
    if candidate["source_ids"] != ["A", "B"]:
        raise AssertionError(candidate["source_ids"])
    if candidate["page_urls"] != ["https://example.com/a", "https://example.com/b"]:
        raise AssertionError(candidate["page_urls"])
    if len(candidate["source_ids"]) != len(candidate["page_urls"]):
        raise AssertionError("source_ids/page_urls length mismatch")
    if candidate["shared_source_count"] != 2 or candidate["shared_media_url"] is not True:
        raise AssertionError(candidate)


_synthetic_shared_provenance()
print("synthetic provenance test: PASS")

media_candidates = build_media_candidates(probe_rows)
shared_lookup = {item["media_url"]: item for item in media_candidates}
for row in probe_rows:
    info = shared_lookup.get(row["media_url"])
    if not info:
        continue
    row["shared_source_count"] = info["shared_source_count"]
    row["shared_media_url"] = info["shared_media_url"]

probe = pd.DataFrame(probe_rows, columns=PROBE_COLUMNS)
print("probe_rows:", len(probe), "unique_media:", len(media_candidates))
probe["probe_status"].value_counts(dropna=False)


synthetic provenance test: PASS
probe_rows: 30 unique_media: 15


probe_status
MEDIA_FOUND_PUBLIC    15
NO_MEDIA_FOUND        15
Name: count, dtype: int64

## 8. Summary + success gates

`SUCCESS_VOV4_MEDIA_PROBE` chỉ khi không có `ERROR` và không có `ERROR_ARTICLE_SCOPE_NOT_FOUND`. Summary tách candidate khỏi media public đã verify. Media URL dùng chung nhiều bài được flag `shared_media_url`, không tự fail.


In [8]:
def write_jsonl(path: Path, rows: list) -> None:
    lines = [json.dumps(row, ensure_ascii=False) for row in rows]
    path.write_text(("\n".join(lines) + "\n") if lines else "", encoding="utf-8")


articles = set(candidates["source_id"])
pages = set(candidates["page_url"])
if set(probe["source_id"]) != articles:
    missing = sorted(articles - set(probe["source_id"]))
    raise RuntimeError(f"Probe manifest missing articles: {missing[:5]}")
if not probe["page_url"].isin(pages).all():
    raise RuntimeError("probe page_url không thuộc candidate Notebook 07")
if not probe["source_id"].isin(articles).all():
    raise RuntimeError("probe source_id không thuộc candidate Notebook 07")

candidate_urls = [item["media_url"] for item in media_candidates]
if len(candidate_urls) != len(set(candidate_urls)):
    raise RuntimeError("media_candidates canonical media_url không unique")
for item in media_candidates:
    if not item["source_ids"] or not item["page_urls"]:
        raise RuntimeError(f"media candidate thiếu provenance: {item['media_url']}")
    if len(item["source_ids"]) != len(item["page_urls"]):
        raise RuntimeError(f"source_ids/page_urls lệch nhau: {item['media_url']}")
    if not set(item["source_ids"]).issubset(articles) or not set(item["page_urls"]).issubset(pages):
        raise RuntimeError(f"provenance không thuộc Notebook 07: {item['media_url']}")

allowed_methods = {
    "", "audio_tag", "video_tag", "source_tag", "og_audio", "og_video",
    "json_ld", "script_config", "inline_script", "data_attribute", "href_src_scan",
}
if not set(probe["discovery_method"]).issubset(allowed_methods):
    raise RuntimeError(f"discovery_method lạ: {set(probe['discovery_method']) - allowed_methods}")

media_only = probe[probe["media_found"].astype(bool)]
public_rows = probe[
    probe["probe_status"].eq("MEDIA_FOUND_PUBLIC") & probe["is_public_direct_url"].astype(bool)
]
articles_with_public = set(public_rows["source_id"])
articles_no_public = articles - articles_with_public
if articles_with_public & articles_no_public:
    raise RuntimeError("public/no-public article sets bị chồng")
if articles_with_public | articles_no_public != articles:
    raise RuntimeError("article sets không phủ đủ source_id Notebook 07")
failures = probe[probe["probe_status"].isin(["ERROR", "ERROR_ARTICLE_SCOPE_NOT_FOUND"])]
n_failures = int(len(failures))
status = "SUCCESS_VOV4_MEDIA_PROBE" if n_failures == 0 else "PARTIAL_VOV4_MEDIA_PROBE"
unique_media = pd.DataFrame(media_candidates) if media_candidates else pd.DataFrame(
    columns=["media_url", "media_type", "media_host", "is_public_direct_url", "probe_status"]
)
if len(unique_media):
    public_unique = unique_media[
        unique_media["probe_status"].eq("MEDIA_FOUND_PUBLIC")
        & unique_media["is_public_direct_url"].astype(bool)
    ]
else:
    public_unique = unique_media
extension_counts = unique_media["media_extension"].value_counts().to_dict() if len(unique_media) and "media_extension" in unique_media.columns else {}
public_hosts = sorted(set(public_unique["media_host"]) - {""}) if len(public_unique) else []

def _nunique_status(status_name: str) -> int:
    subset = probe[probe["probe_status"].eq(status_name) & probe["media_url"].astype(str).ne("")]
    return int(subset["media_url"].nunique()) if len(subset) else 0

summary = {
    "status": status,
    "n_articles_total": int(EXPECTED_ROWS),
    "n_articles_with_public_media": int(len(articles_with_public)),
    "n_articles_no_public_media": int(len(articles_no_public)),
    "n_candidate_media": int(len(media_candidates)),
    "n_candidate_audio": int(unique_media["media_type"].eq("audio").sum()) if len(unique_media) else 0,
    "n_candidate_video": int(unique_media["media_type"].eq("video").sum()) if len(unique_media) else 0,
    "n_public_direct_media": int(len(public_unique)),
    "n_public_audio": int(public_unique["media_type"].eq("audio").sum()) if len(public_unique) else 0,
    "n_public_video": int(public_unique["media_type"].eq("video").sum()) if len(public_unique) else 0,
    "n_protected_or_unsupported": _nunique_status("PROTECTED_OR_UNSUPPORTED"),
    "n_non_media": _nunique_status("NON_MEDIA"),
    "n_low_priority_with_public_media": int(public_rows.loc[public_rows["candidate_priority"].eq("low"), "source_id"].nunique()) if len(public_rows) else 0,
    "n_failures": n_failures,
    "unique_public_media_hosts": public_hosts,
    "media_extension_counts": extension_counts,
    "generated_at_utc": utc_now(),
}

OUT_DIR.mkdir(parents=True, exist_ok=True)
probe.to_csv(OUT_DIR / "vov4_media_probe.csv", index=False, encoding="utf-8-sig")
write_jsonl(OUT_DIR / "vov4_media_probe.jsonl", probe.to_dict(orient="records"))
write_jsonl(OUT_DIR / "media_candidates.jsonl", media_candidates)
(OUT_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
failures_path = OUT_DIR / "failures.jsonl"
if n_failures:
    write_jsonl(failures_path, failures.to_dict(orient="records"))
elif failures_path.exists():
    failures_path.unlink()

print(json.dumps(summary, ensure_ascii=False, indent=2))
if n_failures:
    raise RuntimeError(f"PARTIAL_VOV4_MEDIA_PROBE: {n_failures} failure(s)")
if summary["status"] != "SUCCESS_VOV4_MEDIA_PROBE":
    raise RuntimeError(summary["status"])


{
  "status": "SUCCESS_VOV4_MEDIA_PROBE",
  "n_articles_total": 30,
  "n_articles_with_public_media": 15,
  "n_articles_no_public_media": 15,
  "n_candidate_media": 15,
  "n_candidate_audio": 12,
  "n_candidate_video": 3,
  "n_public_direct_media": 15,
  "n_public_audio": 12,
  "n_public_video": 3,
  "n_protected_or_unsupported": 0,
  "n_non_media": 0,
  "n_low_priority_with_public_media": 3,
  "n_failures": 0,
  "unique_public_media_hosts": [
    "vov4.vov.vn"
  ],
  "media_extension_counts": {
    "mp3": 12,
    "mp4": 3
  },
  "generated_at_utc": "2026-09-24T17:29:04Z"
}


## 9. Quick audit

Chỉ xem manifest. Chưa tải media.


In [9]:
audit_cols = [
    "source_id", "published_date", "candidate_priority", "title",
    "media_found", "media_type", "media_extension", "media_host",
    "discovery_method", "probe_status", "is_public_direct_url", "shared_media_url",
]
display(probe[audit_cols])
print("extension_counts:", summary["media_extension_counts"])
print("public_hosts:", summary["unique_public_media_hosts"])


,source_id,published_date,candidate_priority,title,media_found,media_type,media_extension,media_host,discovery_method,probe_status,is_public_direct_url,shared_media_url
0,VOV4_23A5518B701085121391274F43D7F0A6854AF4A5,2026-09-24,normal,Tơdrong kơtơ̆ng ang năr 24.9.2026,True,audio,mp3,vov4.vov.vn,data_attribute,MEDIA_FOUND_PUBLIC,True,False
1,VOV4_2D39DE6994374745A90B2F8CCC41A43ABF111EF0,2026-09-24,normal,Chánh án Hơnih tơm xek tơlang teh đak Nguyễn V...,False,,,,,NO_MEDIA_FOUND,False,False
2,VOV4_4F3554FC564400008578B69EB55E4D529F2A6E0E,2026-09-24,normal,Hop akŏm pơdrơ̆ng kiơ̆ trong chih tơlĕch Hla b...,False,,,,,NO_MEDIA_FOUND,False,False
3,VOV4_B6C2E19EE2E2E4F8454124A4F5E7D2687EFA78D3,2026-09-24,normal,Hơnhăk choh pơtăm kơjăp truh hăm kon pơlei pơt...,True,audio,mp3,vov4.vov.vn,data_attribute,MEDIA_FOUND_PUBLIC,True,False
4,VOV4_E13F09EA9BD4230FFC98E4F7F7DC8448DFD71104,2026-09-24,normal,Rim tơring kơ Tây Nguyên tơjră hăm 'mi kial đa...,False,,,,,NO_MEDIA_FOUND,False,False
5,VOV4_11AC992B7036F44651DC941869900437CA99DFF2,2026-09-24,normal,Kon pơlei jang mir Đăk Lăk pơđăp gah tơdrong t...,False,,,,,NO_MEDIA_FOUND,False,False
6,VOV4_4E064FE2CE9F07F56C8C6F0B757161248F3E4BCD,2026-09-24,normal,Tổng Bí thư Kơdră teh đak bơ̆n Tô Lâm iung pơm...,False,,,,,NO_MEDIA_FOUND,False,False
7,VOV4_3AACF99A3ADFB771059750FD550F7ED2BD427FB9,2026-09-24,normal,Chánh án Hơnih xek tơlang tơm Nguyễn Văn Quảng...,False,,,,,NO_MEDIA_FOUND,False,False
8,VOV4_18412C2E196AE5AD50187C397998FBDAE4C9EDDD,2026-09-24,normal,"Tổng Bí thư, Kơdră chĕp pơgơ̆r teh đak Tô Lâm ...",False,,,,,NO_MEDIA_FOUND,False,False
9,VOV4_C108541B901E319D5EAC87F59D9FE49489DF174A,2026-09-24,normal,Vang pơjing tơdrong pơchơt pơhiơ̆ Trung thu ăn...,False,,,,,NO_MEDIA_FOUND,False,False


extension_counts: {'mp3': 12, 'mp4': 3}
public_hosts: ['vov4.vov.vn']
